## Install

In [1]:
!pip install -q -U "trl>=0.15" peft transformers datasets accelerate

## Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/spider_data.zip" /content/
!unzip -q /content/spider_data.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/spider_data/dev_gold.sql? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/__MACOSX/spider_data/._dev_gold.sql? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/__MACOSX/spider_data/._dev_gold.sql? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


## Config + Imports

In [3]:
import os, re, time, json, sqlite3, torch
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SPIDER   = "/content/spider_data"
DB_DIR   = f"{SPIDER}/database"
TRAIN_JSON = f"{SPIDER}/train_spider.json"   # TRAIN split — learning happens here
N_TRAIN = 500     # start small; scale up once the loop works

## Verifier + Reward

In [4]:
FORBIDDEN = ("drop","delete","update","insert","alter","create","replace","attach","pragma")

def build_schema_string(db_path, max_tables=20):
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True); cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tables = [r[0] for r in cur.fetchall()][:max_tables]; out = []
    for t in tables:
        cur.execute(f'PRAGMA table_info("{t}")'); out.append(f"{t}({', '.join(r[1] for r in cur.fetchall())})")
    con.close(); return "\n".join(out)

def extract_sql(t):
    t = t if isinstance(t, str) else t[0]["content"]
    m = re.search(r"```sql\s*(.*?)```", t, re.DOTALL|re.IGNORECASE) or re.search(r"```\s*(.*?)```", t, re.DOTALL)
    return re.sub(r"\s+"," ",(m.group(1) if m else t).strip().rstrip(";").strip())

def safe_execute(db_path, sql, timeout=5.0):
    if not sql or any(k in sql.lower() for k in FORBIDDEN): return None
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
    s=time.time(); con.set_progress_handler(lambda:(time.time()-s)>timeout,10000)
    try: return con.cursor().execute(sql).fetchall()
    except Exception: return None
    finally: con.close()

def exec_match(db_path, pred, gold):
    p = safe_execute(db_path, pred)
    if p is None: return 0.0
    g = safe_execute(db_path, gold)
    if g is None: return 0.0
    return 1.0 if set(map(tuple,p))==set(map(tuple,g)) else 0.0

def sql_reward(completions, gold_sql, db_path, **kwargs):
    return [exec_match(dbp, extract_sql(c), gold) for c,gold,dbp in zip(completions, gold_sql, db_path)]

## Build the TRAIN dataset + gate check

In [5]:
raw = json.load(open(TRAIN_JSON))
PROMPT = ("Database schema:\n{schema}\n\nWrite a single SQLite query that answers the question. "
          "Return ONLY the query inside a ```sql code block.\n\nQuestion: {q}")
rows = []
for ex in raw:
    dbp = f"{DB_DIR}/{ex['db_id']}/{ex['db_id']}.sqlite"
    if not os.path.exists(dbp): continue
    msg = [{"role":"user","content": PROMPT.format(schema=build_schema_string(dbp), q=ex["question"])}]
    rows.append({"prompt": msg, "gold_sql": ex["query"], "db_path": dbp})
    if len(rows) >= N_TRAIN: break
dataset = Dataset.from_list(rows)

# GATE: gold must score 1.0 before spending any GPU time
assert exec_match(rows[0]["db_path"], rows[0]["gold_sql"], rows[0]["gold_sql"]) == 1.0, "Verifier broken"
print(f"{len(dataset)} train examples. Verifier OK.")

500 train examples. Verifier OK.


In [6]:
import trl, inspect
from trl import GRPOConfig
print("TRL version:", trl.__version__)
print([f for f in GRPOConfig.__dataclass_fields__ if "length" in f or "prompt" in f or "completion" in f])

TRL version: 1.8.0
['length_column_name', 'max_completion_length', 'vllm_max_model_length', 'mask_truncated_completions', 'log_completions', 'num_completions_to_print', 'log_unique_prompts', 'log_completions_hub_repo']


## Train (GRPO + LoRA)


In [7]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.5 MB/s eta 0:00:00


In [8]:
peft_config = LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = GRPOConfig(
    output_dir="grpo-sql", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    num_generations=8, max_completion_length=256,
    learning_rate=1e-5, beta=0.04, temperature=0.9,
    logging_steps=1, num_train_epochs=1, bf16=True, report_to="none", save_strategy="no",
)
trainer = GRPOTrainer(model=MODEL_ID, args=cfg, train_dataset=dataset,
                      reward_funcs=sql_reward, peft_config=peft_config)
trainer.train()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.596670
2,0.582985
3,0.052937
4,0.118479
5,0.000005
6,0.289737
7,0.288689
8,0.087230
9,0.195454
10,0.000011


TrainOutput(global_step=250, training_loss=0.053633482854434984, metrics={'train_runtime': 1692.6607, 'train_samples_per_second': 0.295, 'train_steps_per_second': 0.148, 'total_flos': 0.0, 'train_loss': 0.053633482854434984, 'epoch': 1.0})

## Merge LoRA and save the trained model

In [9]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("/content/grpo-sql-merged")
trainer.tokenizer.save_pretrained("/content/grpo-sql-merged")
print("Merged model saved. Now point your baseline eval cells (6-8 + EX scorer) at this path.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AttributeError: 'GRPOTrainer' object has no attribute 'tokenizer'

In [10]:
trainer.processing_class.save_pretrained("/content/grpo-sql-merged")
print("Tokenizer saved.")

Tokenizer saved.


In [11]:
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained("/content/grpo-sql-merged")

('/content/grpo-sql-merged/tokenizer_config.json',
 '/content/grpo-sql-merged/chat_template.jinja',
 '/content/grpo-sql-merged/tokenizer.json')

## Load the trained model

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MERGED = "/content/grpo-sql-merged"

tok = AutoTokenizer.from_pretrained(MERGED)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MERGED, torch_dtype=torch.bfloat16, device_map="cuda").eval()
DEVICE = "cuda"

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Build the dev prompts

In [14]:
!nvidia-smi
!pip install -q -U transformers accelerate datasets sqlparse nltk tqdm

Mon Jul 13 20:29:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             56W /  400W |   13216MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [16]:
from tqdm.auto import tqdm

PROMPT = ("Database schema:\n{schema}\n\nWrite a single SQLite query that answers "
          "the question. Return ONLY the query inside a ```sql code block.\n\nQuestion: {question}")
DEV_JSON = f"{SPIDER}/dev.json"
dev = json.load(open(DEV_JSON))
examples = []
for ex in tqdm(dev):
    dbp = f"{DB_DIR}/{ex['db_id']}/{ex['db_id']}.sqlite"
    msg = [{"role":"user","content": PROMPT.format(
        schema=build_schema_string(dbp), question=ex["question"])}]
    text = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    examples.append({"text": text, "gold": ex["query"], "db_id": ex["db_id"], "db_path": dbp})
print(len(examples), "dev examples")

  0%|          | 0/1034 [00:00<?, ?it/s]

1034 dev examples


## Batched generation helper, then the greedy pass

In [17]:
def generate_batched(texts, max_new=256, do_sample=False, num_return=1, temp=0.8, bs=16):
    outs = []
    for i in tqdm(range(0, len(texts), bs)):
        enc = tok(texts[i:i+bs], return_tensors="pt", padding=True,
                  truncation=True, max_length=1536).to(DEVICE)
        kw = dict(max_new_tokens=max_new, pad_token_id=tok.pad_token_id,
                  num_return_sequences=num_return)
        kw.update(dict(do_sample=True, temperature=temp, top_p=0.95) if do_sample else dict(do_sample=False))
        with torch.no_grad():
            g = model.generate(**enc, **kw)
        g = g[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(g, skip_special_tokens=True))
    return outs

greedy = generate_batched([e["text"] for e in examples], do_sample=False, bs=16)
preds  = [extract_sql(g) or "SELECT 1" for g in greedy]

  0%|          | 0/65 [00:00<?, ?it/s]

## Write the prediction and gold files (aligned, one line each)

In [18]:
with open("/content/gold.txt","w") as gf, open("/content/pred.txt","w") as pf:
    for e, p in zip(examples, preds):
        gf.write(f"{e['gold']}\t{e['db_id']}\n")
        pf.write(f"{p}\n")
print("wrote gold.txt and pred.txt")

wrote gold.txt and pred.txt


## pass@1 vs pass@8 diagnostic (subset, ~200 for speed)

In [19]:
K, SUB = 8, 200
sub = examples[:SUB]
samp = generate_batched([e["text"] for e in sub], do_sample=True, num_return=K, temp=0.8, bs=8)
p1 = pk = 0.0
for i, e in enumerate(sub):
    cands = samp[i*K:(i+1)*K]
    p1 += exec_match(e["db_path"], preds[i], e["gold"])
    pk += 1.0 if any(exec_match(e["db_path"], extract_sql(c), e["gold"]) >= 1 for c in cands) else 0.0
print(f"pass@1 (greedy): {p1/len(sub):.3f}")
print(f"pass@{K}        : {pk/len(sub):.3f}")

  0%|          | 0/25 [00:00<?, ?it/s]

pass@1 (greedy): 0.525
pass@8        : 0.670


## Clone the official evaluators

In [20]:
!git clone -q https://github.com/taoyds/spider
!git clone -q https://github.com/taoyds/test-suite-sql-eval

## EX: the reportable execution accuracy + exact match, by difficulty

In [21]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [23]:
TABLES = f"{SPIDER}/tables.json"
print(TABLES, os.path.exists(TABLES))   # should print the path and True

/content/spider_data/tables.json True


In [24]:
!cd spider && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db {DB_DIR} --table {TABLES} --etype all

eval_err_num:1
medium pred: SELECT AVG(Age) AS Average_Age, MIN(Age) AS Minimum_Age, MAX(Age) AS Maximum_Age FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:2
medium pred: SELECT AVG(Age) AS Average_Age, MIN(Age) AS Minimum_Age, MAX(Age) AS Maximum_Age FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:3
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID ORDER BY T1.Age ASC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:4
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN song AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer)
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:5
m

## TS: test-suite accuracy (do this after EX works)

In [25]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [26]:
# After placing test_suite_database.zip in your Drive root:
!cp "/content/drive/MyDrive/test_suite_database.zip" /content/
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head     # folder of <db_id>/<db_id>.sqlite, like the regular DBs

ls: cannot access '/content/test_suite_database': No such file or directory


In [27]:
!unzip -q /content/test_suite_database.zip -d /content/
!ls /content/test_suite_database | head

replace /content/database/voter_1/voter_1v515patch1.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
ls: cannot access '/content/test_suite_database': No such file or directory


In [30]:
!ls /content/                              # what got unzipped, and under what name?
!ls /content/test_suite_database | head

database  grpo-sql-merged  spider		    test-suite-sql-eval
drive	  __MACOSX	   spider_data
gold.txt  pred.txt	   spider_data.zip
grpo-sql  sample_data	   test_suite_database.zip
ls: cannot access '/content/test_suite_database': No such file or directory


In [31]:
!rm -rf /content/ts_db
!mkdir -p /content/ts_db
!unzip -q -o /content/test_suite_database.zip -d /content/ts_db
!ls /content/ts_db

database  __MACOSX


In [32]:
!ls /content/ts_db
!ls /content/ts_db/database | head    # if there's a 'database' wrapper

database  __MACOSX
academic
advising
atis
battle_death
car_1
concert_singer
course_teach
cre_Doc_Template_Mgt
dog_kennels
employee_hire_evaluation


In [33]:
TS_DB = "/content/ts_db/database"      # ← correct this to whatever step 2 showed
!cd test-suite-sql-eval && python evaluation.py --gold /content/gold.txt --pred /content/pred.txt --db {TS_DB} --table {TABLES} --etype exec

                     easy                 medium               hard                 extra                all                 
count                248                  446                  174                  166                  1034                
=====================   EXECUTION ACCURACY     =====================
execution            0.806                0.527                0.351                0.247                0.519               


In [34]:
!cd test-suite-sql-eval && python evaluation.py \
    --gold /content/gold.txt --pred /content/pred.txt \
    --db {DB_DIR} --table {TABLES} --etype exec

                     easy                 medium               hard                 extra                all                 
count                248                  446                  174                  166                  1034                
=====================   EXECUTION ACCURACY     =====================
execution            0.827                0.628                0.454                0.343                0.601               


## Save everything to Drive

In [35]:
import shutil, json, datetime

# timestamped folder so reruns don't overwrite
stamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = f"/content/drive/MyDrive/spider_grpo/{stamp}"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) the prediction + gold files (needed to re-score anytime, no GPU required)
for f in ["/content/gold.txt", "/content/pred.txt"]:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)

# 2) the raw greedy generations (so you never have to regenerate)
with open(f"{SAVE_DIR}/raw_greedy.json", "w") as fh:
    json.dump(greedy, fh)

# 3) the pass@k diagnostic numbers + run config
summary = {
    "model": MODEL_ID,
    "n_dev": len(examples),
    "pass@1_greedy": round(p1/len(sub), 4),
    "pass@8":        round(pk/len(sub), 4),
    "passk_subset":  len(sub),
    "timestamp": stamp,
    "note": "EX/TS come from the official evaluators — paste their printed numbers below",
    "EX_by_difficulty": "FILL IN from cell 11 output",
    "TS_by_difficulty": "FILL IN from cell 12 output",
}
with open(f"{SAVE_DIR}/summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

print("Saved to:", SAVE_DIR)
print(os.listdir(SAVE_DIR))

Saved to: /content/drive/MyDrive/spider_grpo/20260713_205403
['gold.txt', 'pred.txt', 'raw_greedy.json', 'summary.json']
